# Model Governance Review Assistant - Step-by-Step Demo

This notebook is intentionally separate from the Streamlit app and FastAPI backend.

Goal of the demo:
- Read a complex PDF.
- Convert every page to text.
- Detect pages that probably contain tables, charts, or figures.
- Render only those pages as images.
- Ask GPT-5 vision for the crop box around the relevant visual.
- Store normal text chunks in in-memory Chroma for RAG.
- Answer from text chunks only.
- Attach cropped images only when their nearby text fuzzy-matches the selected chunks with enough confidence.

The important design principle is separation:
the LLM answer is generated from retrieved text, while images/tables are attached afterward as trust evidence.


## 1. Install and Import Libraries

Libraries used here:

- `pypdfium2`: opens PDFs, extracts page text, and renders pages as images.
- `Pillow`: crops and saves rendered page images.
- `python-dotenv`: loads `.env` values like `OPENAI_API_KEY` and model names.
- `langchain-text-splitters`: splits page text into retrieval-sized chunks.
- `chromadb`: in-memory vector store for the RAG text chunks.
- `langchain-openai`: calls GPT-5 for answer generation and optional vision-based crop detection.

The install cell is safe to skip if your environment already has these packages.


In [29]:
# Run this only if dependencies are not installed yet.
%pip install pypdfium2 pillow python-dotenv chromadb langchain langchain-openai langchain-text-splitters

Note: you may need to restart the kernel to use updated packages.



[notice] A new release of pip is available: 26.0.1 -> 26.2.1
[notice] To update, run: python.exe -m pip install --upgrade pip


In [30]:
from pathlib import Path
from dotenv import load_dotenv

# load_dotenv() reads key-value pairs from .env into environment variables.
# Example values used later:
# - OPENAI_API_KEY: required only for GPT-5 calls
# - OPENAI_TEXT_MODEL: model used for final text answer
# - OPENAI_VISION_MODEL: model used for page-image crop detection
load_dotenv()

# Path.cwd() changes depending on where the notebook kernel is started.
# If the current directory is notebooks/, the project root is one level up.
# Otherwise, we assume the current directory is already the project root.
PROJECT_ROOT = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()

# The demo reads the first PDF inside docs/.
# This avoids hardcoding sample.pdf and works with the PDF you placed in docs/.
DOCS_DIR = PROJECT_ROOT / "docs"
pdf_files = sorted(DOCS_DIR.glob("*.pdf"))
if not pdf_files:
    raise FileNotFoundError(f"No PDF files found in {DOCS_DIR}")

PDF_PATH = pdf_files[0]

# All notebook-generated crops are written here.
# Keeping artifacts separate makes it easy to inspect or delete them later.
ARTIFACT_DIR = PROJECT_ROOT / "artifacts" / "notebook_demo"
ARTIFACT_DIR.mkdir(parents=True, exist_ok=True)

PDF_PATH, ARTIFACT_DIR


(WindowsPath('d:/Users data/Documents/Project-8_Document_Intelligence/docs/Indo MIM - IC -- HSIE-202609181612588425872.pdf'),
 WindowsPath('d:/Users data/Documents/Project-8_Document_Intelligence/artifacts/notebook_demo'))

## 2. Parse Every Page to Text

Why we do this first:
- RAG should search text, not images.
- Text extraction is cheaper than sending every page to an LLM.
- We can use page text to decide which pages are worth rendering as images.

Key API calls:

- `pdfium.PdfDocument(str(PDF_PATH))`
  Opens the PDF file.

- `range(len(pdf))`
  Iterates through every page index. PDF page indexes start at 0, but humans expect page 1, page 2, etc.

- `page.get_textpage()`
  Creates a text extraction object for one PDF page.

- `get_text_range()`
  Extracts all text from that page.

- `text_page.close()`
  Releases native PDF resources. This matters because PDF libraries keep file/page handles open.


In [31]:
import pypdfium2 as pdfium

def extract_page_text(page):
    # get_textpage() creates a text extraction helper for this PDF page.
    # We keep this as a separate function so the cleanup is always consistent.
    text_page = page.get_textpage()
    try:
        # get_text_range() returns the full page text.
        # If a PDF page is scanned/image-only, this may return an empty string.
        return text_page.get_text_range() or ""
    finally:
        # Always close the text page object to release PDFium resources.
        text_page.close()

# Open the PDF selected in the setup cell.
# str(PDF_PATH) is used because pypdfium2 expects a filesystem path string.
pdf = pdfium.PdfDocument(str(PDF_PATH))

pages = []
for page_index in range(len(pdf)):
    # page_index is 0-based; page_number is 1-based for user-facing citations.
    page_number = page_index + 1
    page = pdf[page_index]
    text = extract_page_text(page)

    # We store page_number and text together because every later step needs page-level provenance.
    pages.append({"page_number": page_number, "text": text})

# Show total pages and a short preview of page 1.
len(pages), pages[0]["text"][:500] if pages else "No pages parsed"


c:\Users\Shubhankit\AppData\Local\Programs\Python\Python313\Lib\site-packages\pypdfium2\_helpers\textpage.py:80: UserWarning: get_text_range() call with default params will be implicitly redirected to get_text_bounded()
  warnings.warn("get_text_range() call with default params will be implicitly redirected to get_text_bounded()")


(42, 'Sector Thematic\r\nIndia Alcoholic Beverages')

## 3. Detect Pages Likely to Contain Tables or Images

We do not want to render and send every page image to GPT-5.
Instead, we first use cheap text heuristics to shortlist likely visual pages.

Signals:

- `VISUAL_TERMS`
  Looks for words like table, figure, chart, graph, exhibit, matrix.

- `NUMERIC_ROW`
  Looks for lines with many numbers. This often indicates financial tables or comparison grids.

- `classify_page(text)`
  Returns two things:
  1. `likely_visual`: boolean flag used to decide whether to render/crop this page.
  2. `reasons`: explainable labels showing why the page was selected.

This is a cost-control step: regex first, LLM only for candidate visual pages.


In [32]:
import re

# re.I makes the regex case-insensitive.
# The word-boundary \b prevents partial matches inside unrelated words.
VISUAL_TERMS = re.compile(
    r"\b(table|figure|fig\.?|chart|graph|diagram|exhibit|appendix|matrix|heatmap)\b",
    re.I,
)

# This pattern means: find a line with at least four numeric-looking values.
# Examples: 10 20 30 40, 12.5 13.0 9.8 7.1, 10% 20% 30% 40%.
NUMERIC_ROW = re.compile(r"(?:\b\d+(?:\.\d+)?%?\b\D*){4,}")

def classify_page(text):
    reasons = []

    # Normalize whitespace so regex matching is less sensitive to PDF line breaks.
    normalized = " ".join(text.split())

    # Signal 1: explicit visual/table/chart words.
    if VISUAL_TERMS.search(normalized):
        reasons.append("visual_keyword")

    # Signal 2: multiple numeric-heavy lines.
    # A single numeric line may be a sentence; two or more is more table-like.
    lines = [line.strip() for line in text.splitlines() if line.strip()]
    numeric_rows = sum(1 for line in lines if NUMERIC_ROW.search(line))
    if numeric_rows >= 2:
        reasons.append("dense_numeric_rows")

    return bool(reasons), reasons

# Add classification results back onto each page dictionary.
for page in pages:
    page["likely_visual"], page["visual_reasons"] = classify_page(page["text"])

# Preview the first 10 pages that will be sent to the visual extraction path.
[{"page": p["page_number"], "reasons": p["visual_reasons"]} for p in pages if p["likely_visual"]][:10]


[{'page': 2, 'reasons': ['dense_numeric_rows']},
 {'page': 3, 'reasons': ['dense_numeric_rows']},
 {'page': 4, 'reasons': ['visual_keyword']},
 {'page': 5, 'reasons': ['visual_keyword']},
 {'page': 6, 'reasons': ['visual_keyword']},
 {'page': 7, 'reasons': ['visual_keyword']},
 {'page': 8, 'reasons': ['visual_keyword', 'dense_numeric_rows']},
 {'page': 9, 'reasons': ['visual_keyword', 'dense_numeric_rows']},
 {'page': 10, 'reasons': ['visual_keyword', 'dense_numeric_rows']},
 {'page': 11, 'reasons': ['visual_keyword', 'dense_numeric_rows']}]

## 4. Render Page Image, Ask GPT-5 for the Crop Box, Then Crop

This step creates visual evidence without letting visuals influence answer generation.

Detailed flow:

1. Render only pages that passed `likely_visual=True`.
2. Convert the page image to base64 so it can be sent to GPT-5 vision.
3. Ask GPT-5 for a normalized bounding box:
   - `left`, `top`, `right`, `bottom`
   - each value must be between 0 and 1
   - values are relative to the full rendered page image
4. Convert normalized coordinates back into pixel coordinates.
5. Crop the image with Pillow.
6. Store:
   - page number
   - visual kind
   - crop path
   - nearby text
   - crop box
   - crop reason

Important: if no API key is available, or if the model returns invalid JSON, the notebook uses a simple fallback body crop.


In [33]:
import base64
import json
import os
import re
from io import BytesIO

from langchain_core.messages import HumanMessage
from langchain_openai import ChatOpenAI
from PIL import Image

def infer_visual_kind(text):
    # This label is only for organization and captions.
    # It does not decide the answer.
    lowered = text.lower()
    if "table" in lowered or "matrix" in lowered:
        return "table"
    if "chart" in lowered or "graph" in lowered:
        return "chart"
    if "figure" in lowered or "diagram" in lowered:
        return "figure"
    return "unknown"

def nearby_key_text(text, max_chars=900):
    # We keep nearby/cue text with each crop.
    # Later, this text is fuzzy-matched against retrieved chunks.
    # max_chars limits noise and keeps matching focused.
    lines = [line.strip() for line in text.splitlines() if line.strip()]
    cues = [
        line
        for line in lines
        if any(token in line.lower() for token in ["table", "figure", "chart", "graph", "diagram"])
    ]
    return " ".join((cues[:6] or lines[:10]))[:max_chars]

def image_to_data_url(image):
    # GPT vision accepts image URLs or base64 data URLs.
    # We use an in-memory PNG because the crop-locator model needs to see the rendered page.
    buffer = BytesIO()
    image.save(buffer, format="PNG")
    encoded = base64.b64encode(buffer.getvalue()).decode("ascii")
    return f"data:image/png;base64,{encoded}"

def fallback_body_crop_box(width, height):
    # Fallback when GPT-5 vision is unavailable or returns invalid JSON.
    # The numbers remove margins/header/footer and keep the central body.
    # Return value shape:
    #   ((left_px, top_px, right_px, bottom_px), reason)
    return (int(width * 0.06), int(height * 0.14), int(width * 0.94), int(height * 0.86)), "heuristic_fallback"

def parse_llm_crop_box(content, width, height):
    # The model is instructed to return JSON only, but this regex is defensive.
    # It extracts the first JSON object even if the model adds surrounding text.
    match = re.search(r"\{.*\}", content, flags=re.DOTALL)
    if not match:
        return None

    payload = json.loads(match.group(0))

    # Model returns normalized coordinates between 0 and 1.
    # We clip values to protect against small model formatting mistakes.
    left = max(0.0, min(1.0, float(payload["left"])))
    top = max(0.0, min(1.0, float(payload["top"])))
    right = max(0.0, min(1.0, float(payload["right"])))
    bottom = max(0.0, min(1.0, float(payload["bottom"])))

    # Reject invalid boxes where the right/bottom edge is before the left/top edge.
    if right <= left or bottom <= top:
        return None

    # Convert normalized coordinates into pixel coordinates.
    box = (int(left * width), int(top * height), int(right * width), int(bottom * height))
    return box, payload.get("reason", "llm_selected_crop")

def llm_crop_box(page_image, page_text, visual_kind):
    # If OPENAI_API_KEY is not set, this demo still runs using the fallback crop.
    if not os.getenv("OPENAI_API_KEY"):
        return fallback_body_crop_box(*page_image.size)

    width, height = page_image.size

    # OPENAI_VISION_MODEL defaults to gpt-5 in .env.
    # We do not pass temperature because GPT-5 may require the default.
    llm = ChatOpenAI(model=os.getenv("OPENAI_VISION_MODEL", "gpt-5"))

    prompt = f"""
You are locating visual evidence in a PDF page image.
Find the tight bounding box around the main {visual_kind} evidence on this page.
Prefer a table, chart, graph, figure, or exhibit that matches the nearby text.

Return only JSON with this schema:
{{"left":0.0,"top":0.0,"right":1.0,"bottom":1.0,"reason":"short reason"}}

Coordinates must be normalized from 0 to 1 relative to image size.
Do not include markdown.

Page image size: {width}x{height}
Nearby text:
{page_text[:1600]}
"""

    # HumanMessage contains both text instructions and the page image.
    response = llm.invoke([
        HumanMessage(content=[
            {"type": "text", "text": prompt},
            {"type": "image_url", "image_url": {"url": image_to_data_url(page_image)}},
        ])
    ])

    try:
        parsed = parse_llm_crop_box(str(response.content), width, height)
    except Exception:
        parsed = None

    return parsed if parsed else fallback_body_crop_box(width, height)

visual_evidence = []
for page_info in pages:
    # Only render likely visual pages. This avoids unnecessary image processing and LLM calls.
    if not page_info["likely_visual"]:
        continue

    page = pdf[page_info["page_number"] - 1]

    # scale=2.0 renders at roughly 2x resolution.
    # Higher scale improves visual readability, but increases image size and LLM payload.
    bitmap = page.render(scale=2.0)
    image = bitmap.to_pil()
    bitmap.close()

    kind = infer_visual_kind(page_info["text"])
    key_text = nearby_key_text(page_info["text"])

    # The LLM returns the crop box. Fallback returns a central body crop.
    box, crop_reason = llm_crop_box(image, key_text, kind)

    # Pillow crop box format: (left, top, right, bottom), measured in pixels.
    crop = image.crop(box)

    image_path = ARTIFACT_DIR / f"page_{page_info['page_number']}_{kind}.png"
    crop.save(image_path)

    visual_evidence.append({
        "page_number": page_info["page_number"],
        "kind": kind,
        "image_path": image_path,
        "key_text": key_text,
        "bbox": box,
        "crop_reason": crop_reason,
    })

visual_evidence[:3]


[{'page_number': 2,
  'kind': 'chart',
  'image_path': WindowsPath('d:/Users data/Documents/Project-8_Document_Intelligence/artifacts/notebook_demo/page_2_chart.png'),
  'key_text': 'geographies, and potential M&A-led optionality. Non-MIM segments such as',
  'bbox': (843, 754, 1137, 998),
  'crop_reason': 'Shareholding pattern table is the main populated data exhibit; no chart directly matches the nearby growth discussion.'},
 {'page_number': 3,
  'kind': 'unknown',
  'image_path': WindowsPath('d:/Users data/Documents/Project-8_Document_Intelligence/artifacts/notebook_demo/page_3_unknown.png'),
  'key_text': 'Page | 2 Indo-MIM: Initiating Coverage ▪ Strong structural tailwinds coming from next-gen innovation industries: Based on the current industry dynamics, IML sits at the intersection of the fastest\x02growing global innovation verticals: 1. Humanoids and robotics - Requires miniaturized, high strength, complex metallic components; MIM is preferred for scaling. 2. Satellite interne

## 5. Chunk Text for Retrieval

RAG works better when long pages are broken into focused chunks.

Parameters:

- `chunk_size=1200`
  Maximum characters per chunk. Large enough for governance context, small enough for targeted retrieval.

- `chunk_overlap=180`
  Repeats the last 180 characters into the next chunk so important sentences are not lost at boundaries.

- `separators`
  Tells LangChain where to split first:
  1. paragraphs
  2. new lines
  3. sentence-ish boundary `. `
  4. spaces
  5. final fallback at character level

Each chunk keeps:
- `id`: unique chunk id
- `page_number`: citation and visual matching
- `text`: content sent into retrieval and answer generation


In [34]:
from langchain_text_splitters import RecursiveCharacterTextSplitter

splitter = RecursiveCharacterTextSplitter(
    # Maximum number of characters per chunk.
    chunk_size=1200,

    # Number of characters repeated between neighboring chunks.
    # This reduces boundary loss when a key sentence spans two chunks.
    chunk_overlap=180,

    # Split order: prefer clean semantic boundaries before falling back to smaller units.
    separators=["\n\n", "\n", ". ", " ", ""],
)

chunks = []
for page in pages:
    # Collapse messy PDF whitespace into normal spaces.
    clean_text = " ".join(page["text"].split())

    # split_text returns a list of chunk strings for this page.
    for idx, chunk_text in enumerate(splitter.split_text(clean_text), start=1):
        chunks.append({
            "id": f"p{page['page_number']}:c{idx}",
            "page_number": page["page_number"],
            "text": chunk_text,
        })

len(chunks), chunks[0] if chunks else None


(108,
 {'id': 'p1:c1',
  'page_number': 1,
  'text': 'Sector Thematic India Alcoholic Beverages'})

## 6. Build In-Memory Chroma Retrieval

Chroma is the vector database used for normal text RAG in this demo.

Important arguments:

- `chromadb.EphemeralClient()`
  Creates an in-memory Chroma client. Nothing is persisted to disk.

- `create_collection(name=...)`
  Creates a temporary collection. The `uuid` suffix prevents duplicate-name errors when the cell is rerun.

- `ids`
  Stable identifiers for each chunk.

- `documents`
  The text Chroma embeds and searches.

- `metadatas`
  Extra page-level information kept alongside each chunk.

- `query_texts=[query]`
  The user question.

- `n_results`
  How many top chunks to retrieve.

Visual matching is separate:
after Chroma selects top chunks, fuzzy matching compares the selected chunk text with each crop's `key_text`.


In [35]:
import uuid
from difflib import SequenceMatcher

import chromadb

# EphemeralClient keeps the vector database in memory.
# Good for notebooks and demos because it has no external service dependency.
chroma_client = chromadb.EphemeralClient()

# Chroma collection names must be unique inside a client.
# uuid4 prevents duplicate collection errors when rerunning this cell.
chunk_collection = chroma_client.create_collection(name=f"notebook_governance_chunks_{uuid.uuid4().hex}")

# Local lookup table so we can recover the full chunk dict from a Chroma result id.
chunk_by_id = {chunk["id"]: chunk for chunk in chunks}

chunk_collection.add(
    # ids: unique IDs that Chroma returns during search.
    ids=[chunk["id"] for chunk in chunks],

    # documents: the actual text Chroma embeds and retrieves.
    documents=[chunk["text"] for chunk in chunks],

    # metadatas: optional structured attributes. Page number is useful for citations.
    metadatas=[{"page_number": chunk["page_number"]} for chunk in chunks],
)

def search_chunks(query, limit=5):
    # query: user's natural-language question.
    # limit: top-k chunks to return.
    # n_results must not exceed the number of indexed chunks.
    results = chunk_collection.query(query_texts=[query], n_results=min(limit, len(chunks)))

    # Chroma returns nested lists because it supports batching multiple queries.
    # We use [0] because this notebook sends exactly one query.
    ids = results["ids"][0]
    distances = results["distances"][0]

    retrieved = []
    for chunk_id, distance in zip(ids, distances):
        chunk = chunk_by_id[chunk_id]

        # Chroma gives a distance where lower is better.
        # Convert it into a simple score where higher is better for display/matching.
        score = 1.0 / (1.0 + float(distance))
        retrieved.append({**chunk, "score": score})
    return retrieved

def normalize_text(text):
    # Lowercase and normalize whitespace before fuzzy matching.
    return " ".join(text.lower().split())

def token_overlap(left, right):
    # Token-overlap score:
    # how many meaningful words from the crop key text appear in the selected chunk.
    left_tokens = {token for token in left.split() if len(token) > 2}
    right_tokens = {token for token in right.split() if len(token) > 2}
    if not left_tokens or not right_tokens:
        return 0.0
    return len(left_tokens & right_tokens) / len(left_tokens)

def visual_chunk_score(visual, chunk):
    # This score decides whether a cropped image/table should be attached.
    # It does NOT decide answer content.
    visual_text = normalize_text(visual["key_text"])
    chunk_text = normalize_text(chunk["text"])
    if not visual_text or not chunk_text:
        return 0.0

    # Same-page visuals are more likely to support that chunk.
    page_bonus = 0.15 if visual["page_number"] == chunk["page_number"] else 0.0

    # overlap catches shared terms.
    overlap = token_overlap(visual_text, chunk_text)

    # SequenceMatcher catches rough phrase-level similarity.
    # We cap to 1200 chars so very long chunks do not dominate runtime.
    sequence = SequenceMatcher(None, visual_text[:1200], chunk_text[:1200]).ratio()

    # Weighted score:
    # - 75% token overlap
    # - 25% sequence similarity
    # - plus small same-page bonus
    return min(1.0, page_bonus + (0.75 * overlap) + (0.25 * sequence))

# Confidence threshold.
# Only visuals with score >= this value are attached to the answer.
# Raise this to attach fewer, higher-confidence images.
# Lower this to attach more images, but with higher risk of irrelevant attachments.
VISUAL_MATCH_MIN_SCORE = 0.35

def attach_visuals_to_chunks(selected_chunks, limit=3, min_score=VISUAL_MATCH_MIN_SCORE):
    # selected_chunks: the top-k text chunks retrieved from Chroma.
    # limit: maximum number of images/tables to attach.
    # min_score: confidence threshold for attaching a crop.
    matches = []
    for visual in visual_evidence:
        best_score = max((visual_chunk_score(visual, chunk) for chunk in selected_chunks), default=0.0)
        if best_score >= min_score:
            matches.append({**visual, "score": best_score})

    # Highest-confidence visuals are shown first.
    return sorted(matches, key=lambda item: item["score"], reverse=True)[:limit]


## 7. Ask a Question and Retrieve Evidence

This cell shows the final retrieval behavior.

Important:
- The query goes only to Chroma.
- Chroma returns top text chunks.
- The answer will be generated from those text chunks only.
- Images are attached afterward through fuzzy matching.
- Attached images do not enter the LLM answer prompt.

Arguments:

- `question`
  The user's natural-language governance question.

- `limit=5` in `search_chunks`
  Retrieve the top 5 text chunks.

- `limit=3` in `attach_visuals_to_chunks`
  Attach up to 3 high-confidence visual crops.


In [36]:
# Example question.
# Replace this with any question you want to ask about the uploaded PDF.
question = "What are the key model governance risks and controls?"

# Step 1: retrieve text evidence from Chroma.
top_chunks = search_chunks(question, limit=5)

# Step 2: attach images/tables only if their source text matches selected chunks.
top_visuals = attach_visuals_to_chunks(top_chunks, limit=3)

# Preview the top text chunks and attached visuals.
top_chunks[:2], top_visuals[:2]


([{'id': 'p3:c2',
   'page_number': 3,
   'text': '. These represent multi-year option value and expand IML’s TAM significantly beyond traditional automotive/medical/consumer categories. ▪ Key risks: (i) A significant portion of its revenue (38%+ in FY26) comes from top 10 customers. (ii) With lack of definitive agreements for sales, the business is run on purchase order basis. (iii) Raw material significantly depends on imports. (iv) Concentration of manufacturing facilities in southern India as well as regional developments could affect operations. (v) External reliance on supply chain logistics. (vi) Currency exchange rate fluctuations may have an adverse impact on the business. (vii) Changing laws, rules, and regulations in India could lead to new compliance requirements and this is uncertain at present',
   'score': 0.42303868733166106},
  {'id': 'p41:c5',
   'page_number': 41,
   'text': '. HSL may from time to time solicit from, or perform broking, or other services for, any com

## 8. Optional: Use OpenAI Through LangChain

This cell generates the final natural-language answer.

Critical design choice:
the prompt receives only `top_chunks`, not `top_visuals`.

Why:
- We want the textual answer to be grounded in retrieved text.
- We do not want image descriptions to change or bias the answer.
- Cropped visuals are attached afterward for stakeholder trust and verification.

Parameters:

- `OPENAI_API_KEY`
  Required for the LLM call.

- `OPENAI_TEXT_MODEL`
  Defaults to `gpt-5`.

- `build_grounded_prompt(question, chunks)`
  Builds the prompt using only retrieved text chunks.


In [37]:
import os
from langchain_openai import ChatOpenAI

def build_grounded_prompt(question, chunks):
    # Convert retrieved chunks into a compact evidence block.
    # Each source includes a page number for citation.
    evidence_text = "\n\n".join(
        f"Source {i+1}: page {item['page_number']}\n{item['text']}"
        for i, item in enumerate(chunks)
    )

    # The prompt explicitly tells the LLM:
    # - answer only from supplied evidence
    # - cite pages
    # - avoid unsupported claims
    return f"""
You are a model governance review assistant.
Answer only from the supplied evidence.
Attach page citations to each insight.
If the evidence is not enough, say that clearly.

Question: {question}

Text evidence:
{evidence_text}
"""

if os.getenv("OPENAI_API_KEY"):
    # Model name is controlled by .env.
    # Do not pass temperature here; GPT-5 may require default sampling settings.
    llm = ChatOpenAI(model=os.getenv("OPENAI_TEXT_MODEL", "gpt-5"))

    # Generate answer from text chunks only.
    response = llm.invoke(build_grounded_prompt(question, top_chunks))
    print(response.content)

    # Visuals are printed after the answer.
    # They were not included in the prompt.
    print("\nAttached visuals, not used for generation:")
    for item in top_visuals:
        print(f"- page {item['page_number']} {item['kind']} -> {item['image_path']} score={item['score']:.3f}")
else:
    print("Set OPENAI_API_KEY in .env to run the LangChain OpenAI answer cell.")


**The evidence is insufficient to assess model governance comprehensively.** It only states that the report *may* have been refined using AI for clarity and readability; it does not identify a model or describe its governance framework. (Source 3, p. 41)

| Key risk or gap | Controls or safeguards evidenced |
|---|---|
| **Limited transparency about AI use.** The conditional disclosure does not establish whether AI was actually used, which tools were used, or whether outputs were validated or reviewed by a human. These controls cannot be verified from the evidence. (Source 3, p. 41) | A limited AI-use disclosure is provided, describing possible use for clarity and readability. No model-specific review or validation control is described. (Source 3, p. 41) |
| **Overreliance or inappropriate use.** The report’s general views do not account for individual investors’ circumstances or risk appetite. (Source 3, p. 41) | The report is designated informational, must not be the sole basis of an

## 9. Close the PDF

The PDF object holds native resources.
Closing it prevents locked files and resource leaks, especially when rerunning notebook cells.


In [38]:
# Close the PDF document after all page-level operations are done.
pdf.close()


True